In [ ]:
%load_ext autoreload
%autoreload 2

## INTRO & SETTINGS

Alongside full confidence *sets*, a fitted p-value function (`calibration_method='p-values'`) also
supports cheaper, more classical-looking outputs: a single **point estimate** (the "Focal" estimate,
$\arg\max_\theta \hat{p}(\theta \mid x)$), one-at-a-time (OAT) **intervals**, and OAT p-value **curves**.
This notebook shows all three, and how to visualize them, using the same 2D Gaussian-mean model as
notebooks 1/2/3.

In [ ]:
# SETTINGS

LIKELIHOOD_COV = 0.01
PRIOR_LOC = 0
PRIOR_COV = 0.1

PARAM_DIM = 2
DATA_DIM = 2
BATCH_SIZE = 1
PARAM_SPACE_BOUNDS = {'low': -1.5, 'high': 1.5}
PARAM_GRID_SIZE = 1_000

CONFIDENCE_LEVEL = 0.90

B = 20_000
B_PRIME = 10_000

## SIMULATE

In [ ]:
import torch

from lf2i.simulator.gaussian import GaussianMean

gm = GaussianMean(
    likelihood_cov=LIKELIHOOD_COV,
    prior='gaussian',
    prior_kwargs={'loc': PRIOR_LOC, 'cov': PRIOR_COV},
    poi_space_bounds=PARAM_SPACE_BOUNDS,
    poi_grid_size=PARAM_GRID_SIZE,
    poi_dim=PARAM_DIM,
    data_dim=DATA_DIM,
    batch_size=BATCH_SIZE,
)

#### Observation

In [ ]:
true_theta = torch.tensor([0.5, -0.3])
x_obs = gm(param=true_theta.reshape(1, PARAM_DIM)).reshape(1, DATA_DIM)
x_obs

## FIT THE P-VALUE FUNCTION

`region_form` controls what `LF2I.inference` returns: `'full'` gives the usual Neyman-inversion
confidence set, while `'point_estimates'`, `'intervals'`, and `'curves'` are all cheaper alternate
views of the *same* fitted p-value function -- fitting it again for each call is wasteful in
practice, but shown separately here for clarity of what each `region_form` returns.

In [ ]:
from lf2i.inference import LF2I
from lf2i.test_statistics import Posterior
from sbi.inference import SNPE

posterior_ts = Posterior(poi_dim=PARAM_DIM, estimator=SNPE())
lf2i = LF2I(test_statistic=posterior_ts)

inference_kwargs = dict(
    x=x_obs,
    evaluation_grid=gm.poi_grid,
    confidence_level=CONFIDENCE_LEVEL,
    calibration_method='p-values',
    calibration_model='nn',
    simulator=gm,
    b=B,
    b_prime=B_PRIME,
)

full_region = lf2i.inference(**inference_kwargs, region_form='full')
point_estimate = lf2i.inference(**inference_kwargs, region_form='point_estimates')
intervals = lf2i.inference(**inference_kwargs, region_form='intervals')
oat_intervals, pvalue_curves, curves_grid = lf2i.inference(**inference_kwargs, region_form='curves')

point_estimate, intervals

## VISUALIZE

#### 1D projections with the point estimate marked

In [ ]:
from lf2i.plot.parameter_regions import plot_parameter_intervals

plot_parameter_intervals(
    full_region[0],
    param_dim=PARAM_DIM,
    point_estimates=[point_estimate[0]],
    true_parameters=[true_theta.numpy()],
    interval_type='projection',
    region_names=['LF2I (p-values)'],
    title=f'{int(CONFIDENCE_LEVEL*100)}% region projected to 1D, with Focal point estimate',
)

#### One-at-a-time (OAT) intervals

In [ ]:
plot_parameter_intervals(
    full_region[0],
    param_dim=PARAM_DIM,
    point_estimates=[point_estimate[0]],
    true_parameters=[true_theta.numpy()],
    interval_type='oat',
    oat_intervals=[oat_intervals[0]],
    region_names=['LF2I (p-values)'],
    title=f'{int(CONFIDENCE_LEVEL*100)}% one-at-a-time intervals',
)

#### Pairplot with diagonal p-value curves

The diagonal panels show the OAT p-value curve for each parameter dimension, with the nominal
confidence level marked -- the interval where the curve exceeds that level is exactly the OAT
interval plotted above.

In [ ]:
from lf2i.plot.parameter_regions import parameter_regions_pairplot

parameter_regions_pairplot(
    full_region[0],
    true_parameter=true_theta.numpy(),
    region_names=['LF2I (p-values)'],
    show_diagonal=True,
    diagonal_type='confidence',
    diagonal_pvalues=pvalue_curves,
    diagonal_grid=curves_grid,
    diagonal_levels=[CONFIDENCE_LEVEL],
)